# Multi-Head Attention

单头 Attention 的流程是：

```
X → (W_Q, W_K, W_V) → Q, K, V → Attention → 输出
```

Multi-Head Attention 只多了三步：

```
X → (W_Q, W_K, W_V) → Q, K, V
  → 拆分成 h 个头（reshape）
  → 每个头独立跑 Attention
  → 拼接（concat）
  → 乘输出投影矩阵 W_O
  → 最终输出
```

就这四个新动作：**拆分 → 并行计算 → 拼接 → 输出投影**。

# 1.线性层映射，生成Q、K、V

In [4]:
import numpy as np 

# 设置超参数
L = 7  # 序列长度
d_model = 8 # 词嵌入维度
h = 2  # 注意力头数(实际常用为8或16)
d_k = d_model // h  # 每个注意力头的维度 = 8 // 2 = 4

np.random.seed(42) # 设置随机种子以获得可重复的结果

# 模拟 embedding 后的输入 X
X = np.random.randn(L, d_model) # (7, 8)
print(f"输入矩阵shape：{X.shape}")

# 三个权重矩阵（实际中是可学习参数）
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)

# print(f"W_Q:\n{W_Q}")
# print(f"W_K:\n{W_K}")
# print(f"W_V:\n{W_V}")

# 线性映射，得到 Q、K、V
Q = X @ W_Q  # (7, 8) @ (8, 8) = (7, 8)
K = X @ W_K  # (7, 8) @ (8, 8) = (7, 8)
V = X @ W_V  # (7, 8) @ (8, 8) = (7, 8)

print(f"Q: {Q.shape}")
print(f"K: {K.shape}")
print(f"V: {V.shape}")


输入矩阵shape：(7, 8)
Q: (7, 8)
K: (7, 8)
V: (7, 8)


# 2.Head 拆分——把 d_model 切成 h 份

In [5]:
# reshape: (L, d_model) -> (L, h, d_k) -> (h, L, d_k)

Q_split = Q.reshape(L, h, d_k)
K_split = K.reshape(L, h, d_k)
V_split = V.reshape(L, h, d_k)

print(f"\n拆分后的Q、K、V矩阵shape：")
print(f"Q_split: {Q_split.shape}")
print(f"K_split: {K_split.shape}")
print(f"V_split: {V_split.shape}")

# 转置为 （h, L, d_k），方便每个注意力头独立计算
Q_head = Q_split.transpose(1,0,2)
K_head = K_split.transpose(1,0,2)
V_head = V_split.transpose(1,0,2)

print(f"\n转置后的Q、K、V矩阵shape：")
print(f"Q_head: {Q_head.shape}")
print(f"K_head: {K_head.shape}")
print(f"V_head: {V_head.shape}")


拆分后的Q、K、V矩阵shape：
Q_split: (7, 2, 4)
K_split: (7, 2, 4)
V_split: (7, 2, 4)

转置后的Q、K、V矩阵shape：
Q_head: (2, 7, 4)
K_head: (2, 7, 4)
V_head: (2, 7, 4)
